# H.E.M.A. — Baseline Model & Site-Holdout Generalization Experiment

**H.E.M.A. — Hematological Evaluation of Machine-learning Accuracy**

This notebook:
1. Loads the cleaned, site-holdout-split CP-AnemiC dataset (`train/`, `val/`, `test_unseen_site/`).
2. Fine-tunes an ImageNet-pretrained EfficientNet-B0 for binary anemia screening.
3. Reports metrics **in-domain** (validation, same 8 hospitals as training) versus
   **unseen-site** (Bolgatanga Regional + Kintampo Municipal, held out entirely) —
   this comparison is the core H.E.M.A. research contribution.

**Run this in Google Colab** (Runtime → Change runtime type → GPU) so pretrained
ImageNet weights can be downloaded and training is fast.


## 1. Setup

In [ ]:
!pip install -q scikit-learn torchvision

import os, zipfile
from google.colab import files

# Upload CP-AnemiC_split_ready.zip (produced by the data-cleaning + site-holdout step)
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/data')

DATA_ROOT = '/content/data/clean_output_split'
print(os.listdir(DATA_ROOT))


## 2. Dataset — RGBA compositing, letterbox resize, augmentation

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score, confusion_matrix, brier_score_loss
import json

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

def composite_and_letterbox(img, target_size=224, bg=(0,0,0)):
    """Flatten RGBA onto a solid background (transparent = crop padding, no
    clinical signal), then resize+pad to a square preserving aspect ratio so the
    thin crescent shape isn't distorted."""
    if img.mode != "RGBA":
        img = img.convert("RGBA")
    background = Image.new("RGB", img.size, bg)
    background.paste(img, mask=img.split()[3])
    img = background
    w, h = img.size
    scale = target_size / max(w, h)
    new_w, new_h = max(1, int(w*scale)), max(1, int(h*scale))
    img = img.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new("RGB", (target_size, target_size), bg)
    canvas.paste(img, ((target_size-new_w)//2, (target_size-new_h)//2))
    return canvas

class HemaConjunctivaDataset(Dataset):
    LABEL_MAP = {"Non-anemic": 0, "Anemic": 1}
    def __init__(self, root_dir, target_size=224, augment=False):
        self.samples = []
        for label_name, label_idx in self.LABEL_MAP.items():
            folder = os.path.join(root_dir, label_name)
            if not os.path.isdir(folder): continue
            for fname in sorted(os.listdir(folder)):
                if fname.lower().endswith(".png"):
                    self.samples.append((os.path.join(folder, fname), label_idx, fname))
        self.target_size = target_size
        aug_ops = []
        if augment:
            # Mild, clinically-plausible augmentation only — avoid aggressive color
            # jitter that could mimic/mask the pallor-vs-redness signal itself.
            aug_ops = [T.RandomHorizontalFlip(p=0.5), T.RandomRotation(degrees=10),
                       T.ColorJitter(brightness=0.15, contrast=0.15)]
        self.transform = T.Compose(aug_ops + [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label, fname = self.samples[idx]
        img = composite_and_letterbox(Image.open(path), self.target_size)
        return self.transform(img), label, fname
    def class_counts(self):
        c = {0:0, 1:0}
        for _, l, _ in self.samples: c[l]+=1
        return c


## 3. Model — EfficientNet-B0, ImageNet-pretrained, two-phase fine-tuning

In [ ]:
def build_model(pretrained=True, num_classes=2):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    net = models.efficientnet_b0(weights=weights)
    in_features = net.classifier[1].in_features
    net.classifier[1] = nn.Linear(in_features, num_classes)
    return net

def set_backbone_trainable(model, trainable):
    for p in model.features.parameters():
        p.requires_grad = trainable


## 4. Metrics — accuracy, sensitivity, specificity, AUROC, calibration (Brier score)

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    accuracy = (tp+tn)/(tp+tn+fp+fn)
    sensitivity = tp/(tp+fn) if (tp+fn)>0 else float('nan')   # recall on Anemic
    specificity = tn/(tn+fp) if (tn+fp)>0 else float('nan')   # recall on Non-anemic
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auroc = float('nan')
    brier = brier_score_loss(y_true, y_prob)
    return {"n": len(y_true), "accuracy": accuracy, "sensitivity_anemic_recall": sensitivity,
            "specificity_nonanemic_recall": specificity, "auroc": auroc, "brier_score": brier,
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}}

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:,1].cpu().numpy()
        all_probs.extend(probs.tolist()); all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_probs)


## 5. Load data & train — Phase 1 (head-only) then Phase 2 (fine-tune)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

train_ds = HemaConjunctivaDataset(os.path.join(DATA_ROOT, 'train'), augment=True)
val_ds   = HemaConjunctivaDataset(os.path.join(DATA_ROOT, 'val'), augment=False)
test_ds  = HemaConjunctivaDataset(os.path.join(DATA_ROOT, 'test_unseen_site'), augment=False)

print("Train:", train_ds.class_counts(), "| Val:", val_ds.class_counts(), "| Unseen-site test:", test_ds.class_counts())

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

counts = train_ds.class_counts()
total = counts[0] + counts[1]
class_weights = torch.tensor([total/(2*counts[0]), total/(2*counts[1])], dtype=torch.float32).to(device)
print("Class weights (Non-anemic, Anemic):", class_weights.tolist())

def train_phase(model, epochs, lr, log_prefix):
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_auroc, best_state, history = -1, None, []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item()*imgs.size(0)
        train_loss = running_loss/len(train_loader.dataset)
        y_true, y_prob = evaluate(model, val_loader, device)
        m = compute_metrics(y_true, y_prob)
        history.append({"epoch": epoch+1, "train_loss": train_loss, **m})
        print(f"{log_prefix} epoch {epoch+1}/{epochs}  loss={train_loss:.4f}  val_acc={m['accuracy']:.3f}  val_auroc={m['auroc']:.3f}")
        if m['auroc'] > best_auroc:
            best_auroc = m['auroc']; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, history

set_seed()
model = build_model(pretrained=True).to(device)

set_backbone_trainable(model, False)
model, hist_head = train_phase(model, epochs=5, lr=1e-3, log_prefix="[head]")

set_backbone_trainable(model, True)
model, hist_finetune = train_phase(model, epochs=10, lr=1e-4, log_prefix="[finetune]")


## 6. Final comparison — in-domain validation vs. unseen-site test\n\n**This is the core H.E.M.A. result.** Test-set metrics are computed exactly once, here, after model selection is already frozen (selected by validation AUROC only).

In [ ]:
y_true_val, y_prob_val = evaluate(model, val_loader, device)
val_final = compute_metrics(y_true_val, y_prob_val)

y_true_test, y_prob_test = evaluate(model, test_loader, device)
test_final = compute_metrics(y_true_test, y_prob_test)

print("=== IN-DOMAIN VALIDATION (same 8 hospitals as training) ===")
print(json.dumps(val_final, indent=2))
print("\n=== UNSEEN-SITE TEST (Bolgatanga + Kintampo, never seen in training) ===")
print(json.dumps(test_final, indent=2))

results = {"architecture": "efficientnet_b0", "pretrained": True, "seed": SEED,
           "in_domain_validation": val_final, "unseen_site_test": test_final,
           "training_history_head_phase": hist_head, "training_history_finetune_phase": hist_finetune}
with open('/content/results.json', 'w') as f:
    json.dump(results, f, indent=2)
torch.save(model.state_dict(), '/content/model_weights.pt')


## 7. Comparison plot — for the Expo submission's required 'performance comparison' image

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['accuracy', 'sensitivity_anemic_recall', 'specificity_nonanemic_recall', 'auroc']
labels_pretty = ['Accuracy', 'Sensitivity\n(Anemic recall)', 'Specificity\n(Non-anemic recall)', 'AUROC']
val_vals = [val_final[m] for m in metrics_to_plot]
test_vals = [test_final[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(x - width/2, val_vals, width, label='In-domain (val)')
ax.bar(x + width/2, test_vals, width, label='Unseen-site (test)')
ax.set_xticks(x); ax.set_xticklabels(labels_pretty)
ax.set_ylim(0,1)
ax.set_ylabel('Score')
ax.set_title('H.E.M.A.: In-Domain vs. Unseen-Site Generalization')
ax.legend()
plt.tight_layout()
plt.savefig('/content/in_domain_vs_unseen_site.png', dpi=200)
plt.show()


## 8. Interpretation checklist (fill in after running)

- State the **absolute gap** between in-domain and unseen-site metrics (e.g. AUROC drop).
- Do **not** claim causation yet — this cell only establishes *whether* a generalization
  gap exists and *how large* it is. Diagnosis (image quality, population, acquisition
  differences) is the next notebook/step, using the `HOSPITAL`, `REGION`, `GENDER`,
  `Age(Months)` columns in `split_assignment.csv`.
- If the gap is small, say so plainly — a null result here is still a valid, reportable
  finding for the Research Depth criterion.
- Report `n` for both sets alongside every metric — 64 (val) and 85 (test) are small
  samples, and confidence intervals will be wide. Consider bootstrapped CIs before
  making strong claims in the final writeup.
